# 03 — Gold: Café Enchancia — One consolidated sales table
**User story:** As a business user, I want all transformed store orders in one reporting table so I can analyze sales across branches.

Destination: **ws_tip_demo.<target_schema>.ec_gold_sales**.

Grain: one row per store_id + transaction_id. Gold combines the selected per-store Silver tables with unionByName, keeps all transformed columns, and adds sold-unit and order-count measures. COMPLETED and CANCELLED orders remain in the table. Cancellations have zero sales and zero sold units. This notebook does not save separate daily or product summary tables.

Run Silver first. Default store = KR_SEOUL. The ALL option requires all seven Silver tables in the selected target schema. A specific store refreshes only that store's rows in the same Gold table. Source prices remain local; reporting uses net_sales_php. No second FX conversion is applied.

The source must contain the full intended batch for each refreshed store. This notebook replaces the store's entire slice, not just a newly arrived day. Run sequentially during the workshop. If groups need isolation, assign separate schemas. Old summary tables from the previous notebook are not automatically deleted.


## Workshop setup and scope
Set **store** and **target_schema** before running, then run the configuration cell and all following cells. Use the same values across Bronze, Silver and Gold. The instructor default schema is `tip_demo_schema`; students use their assigned `tip_group_01` through `tip_group_10`. Schemas and permissions must be provisioned before class. Source files remain in the shared training Volumes, regardless of the target schema.

Run Bronze, then Silver, then Gold for one assigned store. Gold `ALL` is an instructor consolidation option after all seven stores have been loaded in the same target schema. A one-store run creates three tables; a full seven-store run creates 15 tables: seven Bronze, seven Silver, one Gold.

This is a fixed seven-day archive exercise. No direct POS, email or network-drive connection is made by students. Scheduling the 8 a.m. New York report, arrival/completeness monitoring, alerts, incremental history, corrected-file handling and refunds remain follow-up backlog items. Tax is excluded in the fictional data. All original files for the selected store must remain present for a full-batch rerun.


In [ ]:
dbutils.widgets.text("store", "KR_SEOUL", "Store ID, or ALL for all seven stores")

dbutils.widgets.text("target_schema", "tip_demo_schema", "Target schema (use your assigned group schema)")


## 1. Read the selected Silver tables
ALL combines all seven stores; a missing Silver table stops the run before any write. Korea and Seoul remain aliases for KR_SEOUL. The destination name is constant for every store selection.


In [ ]:
from pyspark.sql import functions as F

STORE_IDS = ["PH_LUZON", "PH_VIS", "PH_MIND", "US_NY", "US_QUEENS", "US_NJ", "KR_SEOUL"]
value = dbutils.widgets.get("store").strip().upper()
STORE = {"KOREA": "KR_SEOUL", "SEOUL": "KR_SEOUL"}.get(value, value)
if STORE not in STORE_IDS + ["ALL"]:
    raise ValueError(f"Use one of {STORE_IDS}, or ALL.")
CATALOG = "ws_tip_demo"
SCHEMA = dbutils.widgets.get("target_schema").strip()
import re
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError("target_schema must be a valid assigned schema name.")
if not spark.catalog.databaseExists(f"{CATALOG}.{SCHEMA}"):
    raise ValueError(f"Schema {CATALOG}.{SCHEMA} is unavailable. Ask your instructor to provision it.")
spark.conf.set("spark.sql.session.timeZone", "UTC")
selected_stores = STORE_IDS if STORE == "ALL" else [STORE]
GOLD_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`ec_gold_sales`"
frames = []
for store_id in selected_stores:
    table = f"`{CATALOG}`.`{SCHEMA}`.`ec_silver_{store_id.lower()}`"
    if not spark.catalog.tableExists(table):
        raise ValueError(f"Run Silver for {store_id} first. Missing: {table}")
    frame = spark.table(table)
    assert frame.count() > 0, f"Empty Silver table: {table}"
    assert frame.filter(F.col("store_id").isNull() | (F.col("store_id") != store_id)).count() == 0
    frames.append(frame)
silver = frames[0]
for frame in frames[1:]:
    silver = silver.unionByName(frame)
assert silver.filter(F.col("reporting_currency").isNull() | (F.col("reporting_currency") != "PHP")
                     | F.col("net_sales_php").isNull() | F.col("report_date_ny").isNull()).count() == 0
assert silver.select("store_id", "transaction_id").distinct().count() == silver.count()
assert silver.filter(~F.col("order_status").isin("COMPLETED", "CANCELLED") | F.col("order_status").isNull()).count() == 0
assert silver.filter("order_status = 'CANCELLED' AND net_sales_php != 0").count() == 0
print(f"Stores included: {selected_stores}")


## 2. Prepare the reporting rows
Retain every Silver column, including source currency, local prices, local timestamp, UTC timestamp, New York date, FX rate and PHP sales. Add simple additive measures:

- units_sold: quantity for completed orders; otherwise zero.
- completed_orders: one for COMPLETED; otherwise zero.
- cancelled_orders: one for CANCELLED; otherwise zero.

Do not sum quantity for sales reporting: it also includes cancelled quantities. Use units_sold. Sum net_sales_php for revenue; local monetary columns cannot be summed across currencies.


In [ ]:
gold = (silver
    .withColumn("units_sold", F.when(F.col("order_status") == "COMPLETED", F.col("quantity")).otherwise(0))
    .withColumn("completed_orders", F.when(F.col("order_status") == "COMPLETED", 1).otherwise(0))
    .withColumn("cancelled_orders", F.when(F.col("order_status") == "CANCELLED", 1).otherwise(0)))
source_count = silver.count()
assert gold.count() == source_count
assert gold.select("store_id", "transaction_id").distinct().count() == source_count
assert gold.filter("order_status = 'CANCELLED' AND (net_sales_php != 0 OR units_sold != 0 OR completed_orders != 0)").count() == 0
print(f"Rows to write: {source_count:,}; destination: {GOLD_TABLE}")


## 3. Save ONE Delta table
- ALL: replace the entire Gold table from all seven current Silver tables.
- One store, existing target: use Delta replaceWhere to replace only matching store_id rows. Other stores remain untouched.
- First run, no target: create the table from the selected scope.

No append is used, so rerunning the same store does not duplicate its orders. The store value is validated against a fixed allowlist before it enters the replacement predicate. A missing or empty source is rejected; this is not a store-deletion workflow.


In [ ]:
target_exists = spark.catalog.tableExists(GOLD_TABLE)
if STORE == "ALL":
    gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
elif target_exists:
    (gold.write.format("delta").mode("overwrite")
        .option("replaceWhere", f"store_id = '{STORE}'")
        .saveAsTable(GOLD_TABLE))
else:
    gold.write.format("delta").mode("errorifexists").saveAsTable(GOLD_TABLE)
print(f"Saved {GOLD_TABLE}. Refreshed scope: {STORE}")


## 4. Validate saved records
Compare the selected scope to its transformed Silver inputs, including every column and duplicate multiplicity. When refreshing one store, other stores are intentionally outside this comparison. The final summary shows all stores currently loaded in the shared Gold table.

Rerun with unchanged Silver inputs. The selected store's rows and totals should stay the same. ALL should have 25,824 rows for the unchanged seven-store sample dataset.


In [ ]:
saved = spark.table(GOLD_TABLE)
saved_scope = saved if STORE == "ALL" else saved.filter(F.col("store_id") == STORE)
saved_scope = saved_scope.select(*gold.columns)
assert saved_scope.count() == source_count
assert saved_scope.select("store_id", "transaction_id").distinct().count() == source_count
assert gold.exceptAll(saved_scope).limit(1).count() == 0, "Missing or changed reporting rows."
assert saved_scope.exceptAll(gold).limit(1).count() == 0, "Unexpected reporting rows."
assert saved.select("store_id", "transaction_id").distinct().count() == saved.count(), "Duplicate keys in shared Gold table."
display(saved.groupBy("store_id", "reporting_currency")
    .agg(F.count("*").alias("all_orders"),
         F.sum("completed_orders").alias("completed_orders"),
         F.sum("cancelled_orders").alias("cancelled_orders"),
         F.sum("units_sold").alias("units_sold"),
         F.sum("net_sales_php").alias("net_sales_php"))
    .orderBy("store_id"))
print(f"Gold checks passed. Total rows now in {GOLD_TABLE}: {saved.count():,}")


## 5. Query the single table
These displays are queries only; they create no additional saved tables. They use all stores currently loaded in ec_gold_sales. For a complete seven-store report, run ALL after all Silver loads, or refresh each of the seven stores individually.

These are historical archive results, not an as-of-8-a.m. report. October 18 and 25 New York dates have partial global boundary coverage for the sample archive. October 19–24 have full scheduled global coverage only after all seven complete local-date archives are loaded. No file-arrival completeness monitor is implemented.


In [ ]:
display(saved.groupBy("report_date_ny", "reporting_currency")
    .agg(F.sum("net_sales_php").alias("net_sales_php"),
         F.sum("units_sold").alias("units_sold"),
         F.sum("completed_orders").alias("completed_orders"),
         F.sum("cancelled_orders").alias("cancelled_orders"))
    .orderBy("report_date_ny"))
display(saved.filter("order_status = 'COMPLETED'")
    .groupBy("product_name", "category", "reporting_currency")
    .agg(F.sum("net_sales_php").alias("net_sales_php"), F.sum("units_sold").alias("units_sold"))
    .orderBy(F.desc("net_sales_php")))

# Compare stores and countries by New York reporting day.
display(saved.groupBy("report_date_ny", "store_id", "country", "reporting_currency")
    .agg(F.sum("net_sales_php").alias("net_sales_php"),
         F.sum("units_sold").alias("units_sold"),
         F.sum("completed_orders").alias("completed_orders"))
    .orderBy("report_date_ny", "store_id"))

# Product performance by reporting day. These queries create no extra tables.
display(saved.groupBy("report_date_ny", "product_name", "reporting_currency")
    .agg(F.sum("net_sales_php").alias("net_sales_php"), F.sum("units_sold").alias("units_sold"))
    .orderBy("report_date_ny", F.desc("net_sales_php")))
